In [1]:
import numpy as np
import numpy.typing as npt

try:
    import cupy as cp

    if cp.cuda.runtime.getDeviceCount() > 0:
        xp = cp
        device = "cuda"
    else:
        raise Exception("CUDA is not available")
except (ImportError, Exception):
    xp = np
    device = "cpu"

device

'cuda'

In [2]:
class ReLU:
    _inputs: npt.NDArray = None

    def __init__(self):
        pass

    def __call__(self, inputs: npt.NDArray) -> npt.NDArray:
        self._inputs = inputs

        return xp.maximum(0, inputs)

    def backward(self, grad: npt.NDArray) -> npt.NDArray:
        return xp.where(self._inputs > 0, grad, 0)

In [3]:
class Softmax:
    _outputs: npt.NDArray = None

    def __init__(self):
        pass

    def __call__(self, inputs: npt.NDArray) -> npt.NDArray:
        max_logit = xp.max(inputs, axis=-1, keepdims=True)

        self._outputs = xp.exp(inputs - max_logit) / xp.sum(
            xp.exp(inputs - max_logit), axis=-1, keepdims=True
        )

        return self._outputs

    def backward(self, grad: npt.NDArray) -> npt.NDArray:
        return grad * self._outputs - self._outputs * xp.sum(
            grad * self._outputs, axis=-1, keepdims=True
        )

In [ ]:
class PositionalEncoding:
    def __init__(self, embedding_length: int, scale=1.0):
        self._embedding_length = embedding_length
        self._scale = scale

    def __call__(self, inputs: npt.NDArray) -> npt.NDArray:
        T = inputs.shape[1]

        pos = xp.arange(0, T, dtype=int)[:, xp.newaxis]

        denominator = (
             xp.arange(0, self._embedding_length) // 2 * 2
        ) / self._embedding_length
        denominator = 1 / (10_000**denominator)
        denominator = denominator[xp.newaxis, ...]

        inner = pos * denominator
        pe_sin = xp.sin(inner)
        pe_cos = xp.cos(inner)

        pe_sin[:, xp.arange(1, self._embedding_length, 2)] = 0
        pe_cos[:, xp.arange(0, self._embedding_length, 2)] = 0

        positional_encoding = pe_sin + pe_cos

        return inputs + positional_encoding[xp.newaxis, ...] * self._scale

    def backward(self, grad: npt.NDArray) -> npt.NDArray:
        return grad